# Vector Database SDK - Quick Start Guide

A simple guide to using the Vector Database Python SDK.

**Requirements:**
- API server running on `http://localhost:8000`
- Cohere API key in `.env` file

## Setup

In [27]:
import os
from dotenv import load_dotenv
import cohere
from vector_db.sdk import VectorDBClient

load_dotenv()

client = VectorDBClient(base_url=os.getenv("API_BASE_URL", "http://localhost:8000/api/v1"))
co = cohere.Client(os.getenv("COHERE_API_KEY"))

## Create a Library

In [28]:
library = client.create_library(
    name="My First Library",
    index_type="hnsw",
    distance_metric="cosine"
)

## Create a Document

In [29]:
document = client.create_document(
    library_id=library.id,
    name="Sample Document"
)

## Prepare Some Text Data

In [30]:
texts = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a subset of artificial intelligence",
    "Python is a popular programming language",
    "Vector databases enable semantic search",
    "Natural language processing helps computers understand text"
]

## Generate Embeddings

In [31]:
response = co.embed(
    texts=texts,
    model="embed-english-v3.0",
    input_type="search_document"
)
embeddings = response.embeddings

## Add Chunks

In [32]:
for text, embedding in zip(texts, embeddings):
    client.create_chunk(
        document_id=document.id,
        text=text,
        embedding=embedding
    )

## Search

In [33]:
query = "What is AI?"

query_embedding = co.embed(
    texts=[query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=3
)

for result in results.results:
    print(f"Score: {result.score:.4f}")
    print(f"Text: {result.chunk.text}")
    print()

Score: 0.6615
Text: Machine learning is a subset of artificial intelligence

Score: 0.5946
Text: Natural language processing helps computers understand text

Score: 0.5623
Text: Python is a popular programming language



## Add Metadata

In [34]:
new_chunk = client.create_chunk(
    document_id=document.id,
    text="Databases are a key component in information systems",
    embedding=co.embed(
        texts=["Databases store and organize data efficiently"],
        model="embed-english-v3.0",
        input_type="search_document"
    ).embeddings[0],
    metadata={
        "category": "database",
        "difficulty": "beginner"
    }
)

## Search with Filters

In [36]:
results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=5,
    filters={"category": "database"}
)

for result in results.results:
    print(f"{result.chunk.text}")

Databases are a key component in information systems


## List Libraries

In [37]:
libraries = client.list_libraries()

for lib in libraries.items:
    print(f"{lib.name} - {lib.index_config.index_type}")

My First Library - flat
My First Library - lsh
My First Library - flat
My First Library - hnsw
Benchmark-flat-cosine - flat
My First Library - hnsw


## List Documents

In [38]:
documents = client.list_documents(library_id=library.id)

for doc in documents.items:
    print(doc.name)

Sample Document


## List Chunks

In [39]:
chunks = client.list_chunks(document_id=document.id)

for chunk in chunks:
    print(chunk.text)

The quick brown fox jumps over the lazy dog
Machine learning is a subset of artificial intelligence
Python is a popular programming language
Vector databases enable semantic search
Natural language processing helps computers understand text
Databases are a key component in information systems


## Get a Specific Chunk

In [41]:
chunk = chunks[1]  # Get the AI-related chunk
retrieved_chunk = client.get_chunk(chunk.id)
retrieved_chunk.text

'Machine learning is a subset of artificial intelligence'

## Update a Chunk's Embedding

In [42]:
# Update the chunk with a completely different text and embedding
new_text = "Dogs are loyal pets and great companions"
new_embedding = co.embed(
    texts=[new_text],
    model="embed-english-v3.0",
    input_type="search_document"
).embeddings[0]

updated_chunk = client.update_chunk(
    chunk_id=chunk.id,
    text=new_text,
    embedding=new_embedding
)

## Search Again - See How Results Changed

In [43]:
# Same AI query as before
results = client.search_library(
    library_id=library.id,
    query=query_embedding,
    top_k=3
)

print("Results after update:")
for result in results.results:
    print(f"Score: {result.score:.4f}")
    print(f"Text: {result.chunk.text}")
    print()

Results after update:
Score: 0.5946
Text: Natural language processing helps computers understand text

Score: 0.5623
Text: Python is a popular programming language

Score: 0.5454
Text: Vector databases enable semantic search



## Delete a Chunk

In [45]:
# Delete the NLP-related chunk
nlp_chunk = chunks[4]
client.delete_chunk(nlp_chunk.id)

## Search After Deletion

In [46]:
# Search for text processing
text_query = "How do computers process language?"
text_query_embedding = co.embed(
    texts=[text_query],
    model="embed-english-v3.0",
    input_type="search_query"
).embeddings[0]

results = client.search_library(
    library_id=library.id,
    query=text_query_embedding,
    top_k=5
)

print(f"Found {len(results.results)} results (NLP chunk was deleted):")
for result in results.results:
    print(f"- {result.chunk.text}")

Found 5 results (NLP chunk was deleted):
- Python is a popular programming language
- Vector databases enable semantic search
- Databases are a key component in information systems
- Dogs are loyal pets and great companions
- The quick brown fox jumps over the lazy dog


## Cleanup

In [47]:
client.delete_library(library.id)